# ChatGPT-4o Response Collection
## Thesis Phase II — standalone collection notebook

Collects 30 responses (15 questions × EN + RU) from GPT-4o.
Output: `chatgpt_responses.json` saved to evaluation folder.


In [ ]:
!pip install -q openai tqdm
print(" Done")

✅ Done


In [ ]:
import os, json, time
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
print("Imports OK")

Imports OK ✅


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
BASE     = Path("/content/drive/MyDrive/medical_protocols")
OUT_DIR  = BASE / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
QUESTIONS_JSON = BASE / "question_design_v1" / "final_questions.json"
print(f"OUT_DIR: {OUT_DIR}")

OUT_DIR: /content/drive/MyDrive/medical_protocols/evaluation


In [ ]:

OPENAI_API_KEY = ""   # ← paste here, e.g. "sk-proj-..."

if not OPENAI_API_KEY:
    try:
        from google.colab import userdata
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY") or ""
    except Exception:
        pass

print("OPENAI_API_KEY:", " set" if OPENAI_API_KEY else "MISSING — paste above")

OPENAI_API_KEY: ✅ set


In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical assistant specialising in obstetrics and gynaecology. "
    "Answer the following question based on evidence-based clinical guidelines. "
    "Be specific and include: diagnostic criteria, medication names and dosages "
    "where relevant, gestational age thresholds, contraindications, and "
    "recommended follow-up intervals. "
    "Do not give general health advice — provide protocol-level clinical detail."
)
print("System prompt set")

System prompt set ✅


In [ ]:
questions = json.loads(QUESTIONS_JSON.read_text(encoding="utf-8"))
print(f"Loaded {len(questions)} questions:")
for q in questions:
    print(f"  {q['id']} | {q['question_en'][:70]}...")

Loaded 15 questions:
  A1 | What are the diagnostic criteria for severe preeclampsia, and what blo...
  A2 | What fasting glucose and oral glucose tolerance test thresholds are us...
  A3 | What clinical and laboratory criteria define septic shock in obstetric...
  B1 | What is the recommended first-line antihypertensive therapy for severe...
  B2 | What empirical antibiotic regimen is recommended for obstetric sepsis,...
  B3 | What tocolytic agents are recommended for preterm labour, and what are...
  C1 | What are the stepwise surgical interventions recommended for postpartu...
  C2 | What are the indications for emergency hysterectomy in obstetric patie...
  C3 | What are the clinical signs and immediate management steps for suspect...
  D1 | What antenatal screening tests are recommended during the first trimes...
  D2 | What glucose monitoring targets are recommended for patients with gest...
  D3 | What risk factors identify pregnant women who should receive progester...
  E1 | 

In [ ]:

def query_chatgpt(question: str, system_prompt: str) -> str:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        max_tokens=1024,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ],
    )
    return resp.choices[0].message.content.strip()

print("Testing ChatGPT-4o...")
test = query_chatgpt("What is preeclampsia in one sentence?", SYSTEM_PROMPT)
print(f"Response: {test[:150]}...")
print("query_chatgpt() ready ✅")

Testing ChatGPT-4o...
Response: Preeclampsia is a pregnancy-specific hypertensive disorder characterized by new-onset hypertension (≥140/90 mmHg) and proteinuria (≥300 mg/24 hours) o...
query_chatgpt() ready ✅


In [ ]:
SAVE_PATH = OUT_DIR / "chatgpt_responses.json"

if SAVE_PATH.exists():
    collected = json.loads(SAVE_PATH.read_text(encoding="utf-8"))
    print(f"Resumed: {len(collected)} existing responses found.")
else:
    collected = []
    print("Starting fresh.")

done_keys = {(r["question_id"], r["language"]) for r in collected}
pending   = [(q, lang) for q in questions for lang in ["EN", "RU"]
             if (q["id"], lang) not in done_keys]
print(f"Pending: {len(pending)} / 30 queries")

for q, lang in tqdm(pending, desc="ChatGPT-4o"):
    q_text = q["question_en"] if lang == "EN" else q["question_ru"]
    try:
        response_text = query_chatgpt(q_text, SYSTEM_PROMPT)
    except Exception as e:
        response_text = f"ERROR: {e}"
        print(f" {q['id']} {lang}: {e}")

    collected.append({
        "bot":             "ChatGPT-4o",
        "question_id":     q["id"],
        "stratum":         q.get("stratum", ""),
        "language":        lang,
        "question_text":   q_text,
        "response_text":   response_text,
        "target_protocol": q["target_protocol"],
        "timestamp":       datetime.utcnow().isoformat(),
    })
    SAVE_PATH.write_text(
        json.dumps(collected, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    time.sleep(1.0)

print(f"\n✅ Done! {len(collected)}/30 saved to {SAVE_PATH}")
errors = [r for r in collected if r["response_text"].startswith("ERROR:")]
print(f"Errors: {len(errors)} / {len(collected)}")
if errors:
    for r in errors:
        print(f"  ❌ {r['question_id']} {r['language']}: {r['response_text'][:100]}")
else:
    print("All responses clean ✅")

Resumed: 30 existing responses found.
Pending: 0 / 30 queries


ChatGPT-4o: 0it [00:00, ?it/s]


✅ Done! 30/30 saved to /content/drive/MyDrive/medical_protocols/evaluation/chatgpt_responses.json
Errors: 0 / 30
All responses clean ✅


In [ ]:

UPDATED_QUESTIONS = [
    {
        "question_id": "A1", "stratum": "Diagnosis & Criteria",
        "language": "EN",
        "question_text": "A pregnant woman at 34 weeks presents with blood pressure 168/112 mmHg and proteinuria. According to clinical guidelines, does this meet the criteria for severe preeclampsia and what additional features confirm the diagnosis?",
        "target_protocol": "en_Гипертензивные_состояния_при_беременности.md"
    },
    {
        "question_id": "A1", "stratum": "Diagnosis & Criteria",
        "language": "RU",
        "question_text": "Беременная на сроке 34 недели имеет артериальное давление 168/112 мм рт. ст. и протеинурию. Соответствует ли это критериям тяжёлой преэклампсии согласно клиническим протоколам и какие дополнительные признаки подтверждают диагноз?",
        "target_protocol": "en_Гипертензивные_состояния_при_беременности.md"
    },
    {
        "question_id": "A2", "stratum": "Diagnosis & Criteria",
        "language": "EN",
        "question_text": "A pregnant patient undergoes a 75-g oral glucose tolerance test at 26 weeks. The results are: fasting glucose 5.2 mmol/L, 1-hour 10.1 mmol/L, and 2-hour 8.7 mmol/L. According to clinical guidelines, do these values meet the diagnostic criteria for gestational diabetes mellitus?",
        "target_protocol": "en_Сахарный_диабет_при_беременности,_в_родах_и_послеродовом_периоде.md"
    },
    {
        "question_id": "A2", "stratum": "Diagnosis & Criteria",
        "language": "RU",
        "question_text": "Беременная проходит пероральный глюкозотолерантный тест (75 г) на сроке 26 недель. Результаты: глюкоза натощак 5,2 ммоль/л, через 1 час 10,1 ммоль/л, через 2 часа 8,7 ммоль/л. Соответствуют ли эти показатели диагностическим критериям гестационного сахарного диабета согласно клиническим протоколам?",
        "target_protocol": "en_Сахарный_диабет_при_беременности,_в_родах_и_послеродовом_периоде.md"
    },
    {
        "question_id": "D1", "stratum": "Monitoring & Screening",
        "language": "EN",
        "question_text": "A pregnant woman attends her first antenatal visit at 10 weeks of gestation. According to clinical guidelines, what screening tests should be performed during the first trimester and at what gestational weeks are they recommended?",
        "target_protocol": "en_Антенатальный_уход.md"
    },
    {
        "question_id": "D1", "stratum": "Monitoring & Screening",
        "language": "RU",
        "question_text": "Беременная женщина впервые обращается за дородовым наблюдением на сроке 10 недель. Согласно клиническим протоколам, какие скрининговые исследования должны быть проведены в первом триместре и на каких сроках беременности они рекомендуются?",
        "target_protocol": "en_Антенатальный_уход.md"
    },
    {
        "question_id": "D2", "stratum": "Monitoring & Screening",
        "language": "EN",
        "question_text": "A pregnant woman with gestational diabetes measures fasting glucose levels of 5.6 mmol/L during home monitoring. According to clinical guidelines, is this within the recommended glycaemic targets and what management adjustments may be required?",
        "target_protocol": "en_Сахарный_диабет_при_беременности,_в_родах_и_послеродовом_периоде.md"
    },
    {
        "question_id": "D2", "stratum": "Monitoring & Screening",
        "language": "RU",
        "question_text": "Беременная с гестационным сахарным диабетом измеряет уровень глюкозы натощак 5,6 ммоль/л при домашнем мониторинге. Соответствует ли это рекомендуемым целевым показателям гликемии согласно клиническим протоколам и какие коррективы в лечении могут потребоваться?",
        "target_protocol": "en_Сахарный_диабет_при_беременности,_в_родах_и_послеродовом_периоде.md"
    },
]

SAVE_PATH = OUT_DIR / "chatgpt_responses.json"
existing = json.loads(SAVE_PATH.read_text(encoding="utf-8")) if SAVE_PATH.exists() else []
existing_keys = {(r["question_id"], r["language"]): i for i, r in enumerate(existing)}

print(f"Running {len(UPDATED_QUESTIONS)} updated questions through ChatGPT-4o...\n")

for q in tqdm(UPDATED_QUESTIONS, desc="ChatGPT-4o updated Qs"):
    qid  = q["question_id"]
    lang = q["language"]
    print(f"  {qid} {lang} ...", end=" ", flush=True)
    try:
        response_text = query_chatgpt(q["question_text"], SYSTEM_PROMPT)
    except Exception as e:
        response_text = f"ERROR: {e}"
        print(f"❌ {e}")
    record = {
        "bot":             "ChatGPT-4o",
        "question_id":     qid,
        "stratum":         q["stratum"],
        "language":        lang,
        "question_text":   q["question_text"],
        "response_text":   response_text,
        "target_protocol": q["target_protocol"],
        "timestamp":       datetime.utcnow().isoformat(),
    }
    key = (qid, lang)
    if key in existing_keys:
        existing[existing_keys[key]] = record
        print(f"updated ({len(response_text)} chars)")
    else:
        existing.append(record)
        existing_keys[key] = len(existing) - 1
        print(f"added ({len(response_text)} chars)")
    SAVE_PATH.write_text(json.dumps(existing, ensure_ascii=False, indent=2), encoding="utf-8")
    time.sleep(1.0)

print(f"\nDone. chatgpt_responses.json now has {len(existing)} records.")
errors = [r for r in existing if r["response_text"].startswith("ERROR:")]
if errors:
    for r in errors:
        print(f"{r['question_id']} {r['language']}: {r['response_text'][:100]}")
else:
    print("All updated responses clean")


Running 8 updated questions through ChatGPT-4o...



ChatGPT-4o updated Qs:   0%|          | 0/8 [00:00<?, ?it/s]

  A1 EN ... 

/tmp/ipykernel_309/3295556139.py:79: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":       datetime.utcnow().isoformat(),


updated (2793 chars)


...

  D2 EN ... updated (2575 chars)


ChatGPT-4o updated Qs:  88%|████████▊ | 7/8 [01:28<00:11, 11.77s/it]

  D2 RU ... updated (2049 chars)


ChatGPT-4o updated Qs: 100%|██████████| 8/8 [01:37<00:00, 12.19s/it]


✅ Done. chatgpt_responses.json now has 30 records.
All updated responses clean ✅
